In [1]:
import os
import numpy as np
import time

import torchvision
import torch.nn.functional as F
import torch
from torch import nn

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 5090


In [2]:
# Legacy preprocessing helpers (kept for reference; PIL pipeline used instead)
def process(img, crop_s = 16, interp_mode = "bilinear", out_size = [300, 225]):
    img = img.transpose(1, 0, 2)
    # print(f"Input shape: {img.shape}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img is not None:
        # Crop
        cropped_img = crop(img, crop_s)

        # Interpolation
        scale_factor = 300 / cropped_img.shape[0]
        sigma = scale_factor * 0.5
        cropped_img = interpolate(cropped_img, 5, sigma, int_mode = interp_mode, size = out_size)
        cropped_img = cropped_img.transpose(1, 2, 0)

        # Retransform
        cropped_img = cv2.cvtColor(cropped_img, cv2.COLOR_RGB2BGR)
        cropped_img = cropped_img.transpose(1, 0, 2)

        # Adding to array for saving as .npy
        # print(f"Output shape: {cropped_img.shape}")
        return np.array(cropped_img)

def crop(img, crop_s = 16):
    w, h, c = img.shape

    # Crop x pixels from the top and bottom
    if h > 270:
        img = img[:, crop_s:h - crop_s, :]
    #print(img.shape)
    return img

def interpolate(img, kernel_size = 5, sigma = 0.1, int_mode = "bilinear", size = [300, 225]):
    img = torch.tensor(img)
    # Blur for noise reduc
    blur = torchvision.transforms.GaussianBlur(kernel_size, sigma)
    blured_img = blur(img)
    blured_img = blured_img.transpose(0, 2)
    blured_img = blured_img.transpose(1, 2)
    # print(blured_img.shape)
    interpolated_img = F.interpolate(blured_img.unsqueeze(0), size, mode= int_mode)
    interpolated_img = interpolated_img.squeeze(0)
    # print(interpolated_img.shape)
    return interpolated_img.detach().numpy()

def downsample(data):
    d1 = cv2.pyrDown(data)
    d2 = cv2.pyrDown(d1)
    # d3 = cv2.pyrDown(d2)
    print("Original shape: ", data.shape, "Downsampled shape: ", d2.shape)
    return np.array(d2)

def upsample(data):
    d1 = cv2.pyrUp(data)
    d2 = cv2.pyrUp(d1)
    # d3 = cv2.pyrDown(d2)
    print("Original shape: ", data.shape, "Downsampled shape: ", d2.shape)
    return np.array(d2)


def fungi_collate_fn(batch):
    images, class_ids, toxicities, img_names = [], [], [], []
    for image, (class_id, toxicity), img_name in batch:
        # print(f"Individual image shape: {image.shape}")  # Should be [C, H, W]
        images.append(image)
        class_ids.append(class_id)
        toxicities.append(toxicity)
        img_names.append(img_name)

    images = torch.stack(images)  # Should result in shape [batch_size, C, H, W]
    #print(f"Batch images shape after stacking: {images.shape}")

    class_ids = torch.tensor(class_ids, dtype=torch.long)
    toxicities = torch.tensor(toxicities, dtype=torch.long)

    return images, (class_ids, toxicities), img_names

In [3]:
import torch
import torch.nn as nn
import timm  # PyTorch Image Models

NUM_CLASSES = 2427   # category_id 0-2426, already 0-indexed & continuous
NUM_TOX     = 2      # 0 = non-poisonous, 1 = poisonous

class HybridHead(nn.Module):
    def __init__(self, in_features, num_species_classes, num_toxicity_classes):
        super(HybridHead, self).__init__()
        # Attention mechanism
        self.attention = nn.MultiheadAttention(embed_dim=in_features, num_heads=8,
                                               batch_first=True, dropout=0.1)
        # State Space Model (SSM) component
        self.ssm = nn.LSTM(in_features, in_features, batch_first=True)
        # Fully connected layers for species classification
        self.fc_species = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_species_classes)
        )
        # Fully connected layers for toxicity prediction
        self.fc_toxicity = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_toxicity_classes)
        )

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        ssm_output, _  = self.ssm(attn_output)
        pooled_output  = ssm_output.mean(dim=1)
        species_output  = self.fc_species(pooled_output)
        toxicity_output = self.fc_toxicity(pooled_output)
        return species_output, toxicity_output

class FungiClassifier(nn.Module):
    def __init__(self, num_species_classes=NUM_CLASSES, num_toxicity_classes=NUM_TOX):
        super(FungiClassifier, self).__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        in_features = self.backbone.num_features
        self.head = HybridHead(in_features, num_species_classes, num_toxicity_classes)

    def forward(self, x):
        features = self.backbone(x)
        features = features.unsqueeze(1)
        species_output, toxicity_output = self.head(features)
        return species_output, toxicity_output

model = FungiClassifier(NUM_CLASSES, NUM_TOX)
model.to(device)

# Weighted toxicity loss — handles extreme imbalance (7 762 non-pois vs 57 pois)
tox_weight = torch.tensor([1.0, 7762.0 / 57.0], dtype=torch.float).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

criterion_species  = nn.CrossEntropyLoss()
criterion_toxicity = nn.CrossEntropyLoss(weight=tox_weight)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(f"EfficientNet-B0 feature dim: {model.backbone.num_features}")

Trainable parameters: 26,241,529
EfficientNet-B0 feature dim: 1280


In [4]:
IMG_MEAN  = [0.485, 0.456, 0.406]
IMG_STD   = [0.229, 0.224, 0.225]
IMG_SIZE  = 224
SPLIT_MAP = {'train': 'Train', 'val': 'Val', 'test': 'Test'}

class FungiDataset(Dataset):
    """FungiTastic-FewShot dataset (FungiCLEF 2025).

    Labeled splits (train/val): returns (image, category_id, poisonous, filename)
    Test split:                  returns (image, filename, observationID)
    """
    def __init__(self, data_root, split='train', transform=None):
        self.split   = split
        self.img_dir = os.path.join(data_root, 'images', 'FungiTastic-FewShot',
                                    split, '300p')
        csv_name = f'FungiTastic-FewShot-{SPLIT_MAP[split]}.csv'
        self.df  = pd.read_csv(os.path.join(data_root, 'metadata',
                               'FungiTastic-FewShot', csv_name))

        if transform is not None:
            self.transform = transform
        elif split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.RandomRotation(15),
                transforms.ToTensor(),
                transforms.Normalize(IMG_MEAN, IMG_STD),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(IMG_MEAN, IMG_STD),
            ])

        self.has_labels = 'category_id' in self.df.columns
        if self.has_labels:
            self.df['category_id'] = self.df['category_id'].astype(int)
            self.df['poisonous']   = self.df['poisonous'].astype(int)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        img = self.transform(img)
        if self.has_labels:
            return img, int(row['category_id']), int(row['poisonous']), row['filename']
        return img, row['filename'], str(row['observationID'])


def fungi_collate_fn(batch):
    imgs, cids, tox, fns = zip(*batch)
    return (torch.stack(imgs),
            torch.tensor(cids, dtype=torch.long),
            torch.tensor(tox,  dtype=torch.long),
            list(fns))

def fungi_collate_test(batch):
    imgs, fns, obs = zip(*batch)
    return torch.stack(imgs), list(fns), list(obs)

In [5]:
DATA_ROOT  = 'data'
BATCH_SIZE = 32
NUM_EPOCHS = 30

train_dataset = FungiDataset(DATA_ROOT, 'train')
val_dataset   = FungiDataset(DATA_ROOT, 'val')
test_dataset  = FungiDataset(DATA_ROOT, 'test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, collate_fn=fungi_collate_fn,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, collate_fn=fungi_collate_fn,
                          pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, collate_fn=fungi_collate_test,
                          pin_memory=True)

print(f"Train : {len(train_dataset):5d} images | {len(train_loader):3d} batches")
print(f"Val   : {len(val_dataset):5d} images | {len(val_loader):3d} batches")
print(f"Test  : {len(test_dataset):5d} images | {len(test_loader):3d} batches")
print(f"Classes: {NUM_CLASSES}")

Train :  7819 images | 244 batches
Val   :  2285 images |  72 batches
Test  :  1911 images |  60 batches
Classes: 2427


In [6]:
images, class_ids, toxicities, filenames = next(iter(train_loader))

print(f"Image tensor : {images.shape}  dtype={images.dtype}")
print(f"Class IDs    : min={class_ids.min()}, max={class_ids.max()}")
print(f"Toxicities   : {toxicities.unique().tolist()}")
print(f"Filename[0]  : {filenames[0]}")

assert class_ids.min() >= 0 and class_ids.max() < NUM_CLASSES, "Class ID out of range!"
print("Batch OK ✓")

Image tensor : torch.Size([32, 3, 224, 224])  dtype=torch.float32
Class IDs    : min=31, max=2383
Toxicities   : [0]
Filename[0]  : 0-2238493714.JPG
Batch OK ✓


In [7]:
from sklearn.metrics import confusion_matrix, classification_report

def train_model(model, train_loader, val_loader, optimizer, scheduler=None,
                num_epochs=30, alpha=1.0, beta=0.3):
    history = {k: [] for k in ['train_loss', 'val_loss',
                                'train_acc_sp', 'val_acc_sp',
                                'train_acc_tox', 'val_acc_tox']}
    best_val_acc = 0.0
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    print(f"{'Epoch':>5} {'TLoss':>7} {'VLoss':>7} "
          f"{'T-Sp%':>7} {'V-Sp%':>7} {'T-Tox%':>7} {'V-Tox%':>7}  LR")
    print("-" * 72)

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = correct_species = correct_toxicity = total_samples = 0

        for images, class_ids, toxicities, _ in tqdm(
                train_loader, desc=f'Ep {epoch+1}/{num_epochs} Train',
                leave=False, ncols=80):
            images     = images.to(device)
            class_ids  = class_ids.to(device, dtype=torch.long)
            toxicities = toxicities.to(device, dtype=torch.long)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss_species  = criterion_species(outputs[0], class_ids)
                loss_toxicity = criterion_toxicity(outputs[1], toxicities)
                loss = alpha * loss_species + beta * loss_toxicity

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            bs = images.size(0)
            running_loss     += loss.item() * bs
            correct_species  += (outputs[0].argmax(1) == class_ids).sum().item()
            correct_toxicity += (outputs[1].argmax(1) == toxicities).sum().item()
            total_samples    += bs

        avg_train_loss = running_loss / total_samples
        train_acc_sp   = correct_species  / total_samples * 100
        train_acc_tox  = correct_toxicity / total_samples * 100

        # Validation Phase
        model.eval()
        val_running_loss = val_correct_species = val_correct_toxicity = val_total = 0
        val_preds_sp  = []; val_targets_sp  = []
        val_preds_tox = []; val_targets_tox = []

        with torch.no_grad():
            for images, class_ids, toxicities, _ in tqdm(
                    val_loader, desc=f'Ep {epoch+1}/{num_epochs} Val  ',
                    leave=False, ncols=80):
                images     = images.to(device)
                class_ids  = class_ids.to(device, dtype=torch.long)
                toxicities = toxicities.to(device, dtype=torch.long)

                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    outputs = model(images)
                    loss_s = criterion_species(outputs[0], class_ids)
                    loss_t = criterion_toxicity(outputs[1], toxicities)
                    loss   = alpha * loss_s + beta * loss_t

                bs = images.size(0)
                val_running_loss     += loss.item() * bs
                val_correct_species  += (outputs[0].argmax(1) == class_ids).sum().item()
                val_correct_toxicity += (outputs[1].argmax(1) == toxicities).sum().item()
                val_total            += bs

                val_preds_sp.extend(outputs[0].argmax(1).cpu().numpy())
                val_targets_sp.extend(class_ids.cpu().numpy())
                val_preds_tox.extend(outputs[1].argmax(1).cpu().numpy())
                val_targets_tox.extend(toxicities.cpu().numpy())

        avg_val_loss = val_running_loss / val_total
        val_acc_sp   = val_correct_species  / val_total * 100
        val_acc_tox  = val_correct_toxicity / val_total * 100

        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc_sp'].append(train_acc_sp)
        history['val_acc_sp'].append(val_acc_sp)
        history['train_acc_tox'].append(train_acc_tox)
        history['val_acc_tox'].append(val_acc_tox)

        lr_now = optimizer.param_groups[0]['lr']
        print(f"{epoch+1:5d} {avg_train_loss:7.4f} {avg_val_loss:7.4f} "
              f"{train_acc_sp:7.2f} {val_acc_sp:7.2f} "
              f"{train_acc_tox:7.2f} {val_acc_tox:7.2f}  {lr_now:.1e}")

        if val_acc_sp > best_val_acc:
            best_val_acc = val_acc_sp
            torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                        'val_acc_species': val_acc_sp,
                        'val_acc_toxicity': val_acc_tox},
                       'best_model.pth')
            print(f"  Saved best model (val species acc = {val_acc_sp:.2f}%)")

        if epoch == num_epochs - 1:
            np.save('class_conf.npy', confusion_matrix(val_targets_sp,  val_preds_sp))
            np.save('toxconf.npy',    confusion_matrix(val_targets_tox, val_preds_tox))
            print("\nFinal epoch toxicity report:")
            print(classification_report(val_targets_tox, val_preds_tox,
                                        target_names=['Non-pois.','Poisonous'],
                                        zero_division=0))

    print(f"\nTraining complete. Best val species accuracy: {best_val_acc:.2f}%")
    return history

In [8]:
training_stats = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS
)

# Save the model and statistics
torch.save(model.state_dict(), "multi_task_model.pth")
print("Training completed!")

/tmp/ipykernel_2244213/3417890091.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


Epoch   TLoss   VLoss   T-Sp%   V-Sp%  T-Tox%  V-Tox%  LR
------------------------------------------------------------------------


Ep 1/30 Train:   0%|                                    | 0/244 [00:00<?, ?it/s]/tmp/ipykernel_2244213/3417890091.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
Ep 1/30 Val  :   0%|                                     | 0/72 [00:00<?, ?it/s]/tmp/ipykernel_2244213/3417890091.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


    1  7.9571  7.7800    0.23    0.00   99.08   98.99  3.0e-04


    2  7.5963  7.6351    0.47    0.22   98.90   98.99  3.0e-04
  Saved best model (val species acc = 0.22%)


...

   26  5.8156  7.6511    3.23    1.01   99.04   98.60  1.4e-05


   27  5.7984  7.6409    3.28    1.31   99.12   98.56  8.3e-06


   28  5.7774  7.6696    3.61    0.96   98.82   98.73  4.3e-06


   29  5.7726  7.6426    3.41    1.23   98.85   98.69  1.8e-06


   30  5.7875  7.6668    3.38    1.18   98.92   98.64  1.0e-06

Final epoch toxicity report:
              precision    recall  f1-score   support

   Non-pois.       0.99      1.00      0.99      2262
   Poisonous       0.00      0.00      0.00        23

    accuracy                           0.99      2285
   macro avg       0.49      0.50      0.50      2285
weighted avg       0.98      0.99      0.98      2285


Training complete. Best val species accuracy: 1.36%
Training completed!


In [9]:
print(model)
torch.save(model.state_dict(), 'model_dict.pth')
#Save full model
torch.save(model,'full_mode.pth')

FungiClassifier(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
        

In [10]:
from sklearn.metrics import confusion_matrix, classification_report

def eval(model, loader):
    model.eval()
    val_correct_species = val_correct_toxicity = val_total_samples = 0
    val_predictions_species  = []; val_targets_species  = []
    val_predictions_toxicity = []; val_targets_toxicity = []

    with torch.no_grad():
        for images, class_ids, toxicities, _ in tqdm(loader, desc='Eval', ncols=80):
            images     = images.to(device)
            class_ids  = class_ids.to(device, dtype=torch.long)
            toxicities = toxicities.to(device, dtype=torch.long)

            outputs = model(images)

            _, preds_species  = torch.max(outputs[0], 1)
            _, preds_toxicity = torch.max(outputs[1], 1)
            val_correct_species  += (preds_species  == class_ids).sum().item()
            val_correct_toxicity += (preds_toxicity == toxicities).sum().item()
            val_total_samples    += images.size(0)

            val_predictions_species.extend(preds_species.cpu().numpy())
            val_targets_species.extend(class_ids.cpu().numpy())
            val_predictions_toxicity.extend(preds_toxicity.cpu().numpy())
            val_targets_toxicity.extend(toxicities.cpu().numpy())

    print(f"Species  accuracy : {val_correct_species  / val_total_samples * 100:.2f}%")
    print(f"Toxicity accuracy : {val_correct_toxicity / val_total_samples * 100:.2f}%")

    class_conf = confusion_matrix(val_targets_species,  val_predictions_species)
    toxconf    = confusion_matrix(val_targets_toxicity, val_predictions_toxicity)
    np.save('class_conf.npy', class_conf)
    np.save('toxconf.npy',    toxconf)

    print("\nSpecies classification report:")
    print(classification_report(val_targets_species, val_predictions_species,
                                zero_division=0))
    print("\nToxicity classification report:")
    print(classification_report(val_targets_toxicity, val_predictions_toxicity,
                                target_names=['Non-pois.', 'Poisonous'],
                                zero_division=0))

eval(model, val_loader)

Eval: 100%|█████████████████████████████████████| 72/72 [00:01<00:00, 52.76it/s]

Species  accuracy : 1.18%
Toxicity accuracy : 98.64%

Species classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         3
           4       0.00      0.00      0.00         3
           9       0.43      0.38      0.40         8
          10       0.00      0.00      0.00        12
          11       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         1
          19       0.00      0.00      0.00        13
          29       0.00      0.00      0.00         0
          30       0.00      0.00      0.00         4
          31       0.00      0.00      0.00         1
          37       0.00      0.00      0.00         3
          47       0.00      0.00      0.00         2
          55       0.00      0.00      0.00         2
          59       0.00      0.00      0.00       

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

class_conf = np.load('class_conf.npy')
poi_conf   = np.load('toxconf.npy')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(class_conf, cmap='Blues')
axes[0].set_title('Species Classification Confusion Matrix')
axes[0].set_xlabel('Predicted Class'); axes[0].set_ylabel('True Class')
plt.colorbar(axes[0].images[0], ax=axes[0])

sns.heatmap(poi_conf, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Non-pois.', 'Poisonous'],
            yticklabels=['Non-pois.', 'Poisonous'])
axes[1].set_title('Konfuziós mátrix mérgezőségre')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('viz_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

# Training curves
if 'training_stats' in dir():
    epochs = range(1, len(training_stats['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, training_stats['train_loss'], label='Train')
    axes[0].plot(epochs, training_stats['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, training_stats['train_acc_sp'], label='Train')
    axes[1].plot(epochs, training_stats['val_acc_sp'],   label='Val')
    axes[1].set_title('Species Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, training_stats['train_acc_tox'], label='Train')
    axes[2].plot(epochs, training_stats['val_acc_tox'],   label='Val')
    axes[2].set_title('Toxicity Accuracy (%)'); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle('Training History - FungiClassifier (EfficientNet-B0 + HybridHead)',
                 fontsize=13)
    plt.tight_layout()
    plt.savefig('viz_training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()

In [12]:
# Load best checkpoint
ckpt = torch.load('best_model.pth', map_location=device)
model.load_state_dict(ckpt['model'])
print(f"Loaded best model from epoch {ckpt['epoch']} "
      f"(val species acc = {ckpt['val_acc_species']:.2f}%)")

# Test inference — accumulate top-10 scores per observation
model.eval()
preds_by_obs = {}

with torch.no_grad():
    for images, filenames, obs_ids in tqdm(test_loader,
                                           desc='Test inference', ncols=80):
        images = images.to(device)
        sp_out, _ = model(images)
        probs = torch.nn.functional.softmax(sp_out, dim=1)

        for i, obs_id in enumerate(obs_ids):
            top_k    = torch.topk(probs[i], k=10)
            top_cls  = top_k.indices.cpu().tolist()
            top_conf = top_k.values.cpu().tolist()

            if obs_id not in preds_by_obs:
                preds_by_obs[obs_id] = {}
            for cls, conf in zip(top_cls, top_conf):
                preds_by_obs[obs_id][cls] = preds_by_obs[obs_id].get(cls, 0.0) + conf

# Build submission CSV
rows = []
for obs_id, cls_scores in preds_by_obs.items():
    ranked   = sorted(cls_scores.items(), key=lambda x: x[1], reverse=True)
    pred_str = ' '.join(str(c) for c, _ in ranked[:10])
    rows.append({'observationId': obs_id, 'predictions': pred_str})

import pandas as pd
submission = pd.DataFrame(rows)
submission.to_csv('submission_dl.csv', index=False)
print(f"Submission saved — {len(submission)} observations")
print(submission.head(3).to_string())

Loaded best model from epoch 17 (val species acc = 1.36%)


Test inference: 100%|███████████████████████████| 60/60 [00:04<00:00, 14.00it/s]

Submission saved — 999 observations
  observationId                                     predictions
0    4100099350     318 615 253 2040 2351 884 2119 2406 29 2120
1    4100096393  2136 1971 877 1099 1322 272 2026 651 2188 2321
2    4100103428      1877 379 1939 707 1189 1875 915 161 287 13
